# 03_Generate_AB_Test

Генерация контрольной и тестовой групп на основе `funnel.csv`, расчет ключевых метрик и подготовка итогового датасета для анализа.

In [7]:
import pandas as pd
import numpy as np

np.random.seed(42)

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Финальный проект/funnel.csv')
print(df.head())
print(df.columns.tolist())
print(df.shape)


         date  client_id  clicks  finish_watch  start_watch  views  \
0  2025-04-01       1203       1             0            0      0   
1  2025-04-01       1223       0             0            0      1   
2  2025-04-01       1323       0             0            0      1   
3  2025-04-01       1636       0             1            0      0   
4  2025-04-02       1069       1             0            0      0   

   watch_time  adds_to_fav  
0         0.0            0  
1         0.0            0  
2         0.0            0  
3        19.0            0  
4         0.0            0  
['date', 'client_id', 'clicks', 'finish_watch', 'start_watch', 'views', 'watch_time', 'adds_to_fav']
(100, 8)


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Определяем колонку с идентификатором пользователя
possible = ['client_id','user_id','uid','id']
id_col = None
for c in possible:
    if c in df.columns:
        id_col = c
        break

if id_col is None:
    id_col = df.columns[0]

print('ID column:', id_col)


ID column: client_id


In [10]:
users = pd.DataFrame({id_col: df[id_col].drop_duplicates()})

users = users.sample(frac=1, random_state=42).reset_index(drop=True)
users['ab_group'] = np.where(
    users.index < len(users)/2,
    'control',
    'test'
)

df = df.merge(users, on=id_col, how='left')

print(df['ab_group'].value_counts())


ab_group
control    52
test       48
Name: count, dtype: int64


In [11]:
# Автоматический поиск наиболее типичных колонок
event_col = None
for c in ['event','event_name','action','step']:
    if c in df.columns:
        event_col = c
        break

if event_col is not None:
    metrics = (
        df.groupby('ab_group')[event_col]
          .value_counts()
          .unstack(fill_value=0)
    )
    display(metrics)

    if {'view_banner','click_banner'}.issubset(metrics.columns):
        metrics['CTR'] = metrics['click_banner'] / metrics['view_banner']

    if {'start_watch','finish_watch'}.issubset(metrics.columns):
        metrics['Completion'] = (
            metrics['finish_watch'] / metrics['start_watch']
        )

    display(metrics)
else:
    print('Колонка событий не найдена. Сформированы только A/B-группы.')


Колонка событий не найдена. Сформированы только A/B-группы.


In [13]:
df.to_csv('/content/drive/MyDrive/Colab Notebooks/Финальный проект/final_results_to_analyze.csv', index=False)
print('Файл сохранен: final_results_to_analyze.csv')


Файл сохранен: final_results_to_analyze.csv
